# Stage 3 · finish dense Lego training
In a fresh Kaggle notebook select **T4 ×2** and attach the original private Stage 0 kit and the reviewed `lego-dense-vanilla-s42-step2000.zip`. Run the three code cells. The same seed-42 dense vanilla run continues from update 2,000 to 50,000 on GPU 0. At 6.45 updates/s, 48,000 updates take about **2 h 04 min**, plus setup, checkpoint writes and exports. Held-out evaluation and video export follow in separate stages.

Download each milestone ZIP from Output when it appears, especially before leaving a session. A checkpoint in a live notebook needs a copy on your computer or in a saved Kaggle output to survive session loss. The last cell publishes snapshots at 10,000-step milestones and the chosen stop, with local saves every 500 steps. For a shorter session, change `STOP_AT` in that cell to an absolute update number. To continue, increase it and rerun the cell. In a new session attach your latest Stage 3 ZIP alongside the original two inputs. Both zipped and expanded inputs work.

Send **`lego-dense-vanilla-s42-step50000.zip`** when training finishes. If training stops early, save the latest available snapshot and send its error log.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, tempfile, time, zipfile
import torch
from IPython.display import display, FileLink

inputs = Path("/kaggle/input")
outputs = Path("/kaggle/working")
work = outputs / "recovery-main"
relative_run = Path("runs/lego/dense-vanilla/seed-42")
run = work / relative_run
manifest = work / "manifests/lego.json"
run_id = "08ff8d2fceda43759554aa6470054853"
data_hash = "bdc22418467720a3dbf5555318532caf17c250a05433f8553ae58372f11e89e2"
code_hash = "4b85a7f9d5836739fb562c5fb3ff25fa68b063073bbea539e38ed1130f088eb5"
source_hash = "4babbd7b23849a9d98c578753bb273dc6b943b9e2382dd9955ee271ae23b3a8e"

def mounted(marker, zip_patterns, optional=False):
    roots = [p.parent for p in inputs.rglob(marker)]
    archives = [p for pattern in zip_patterns for p in inputs.rglob(pattern)]
    if optional and not (roots or archives):
        return None
    if len(roots) + len(archives) != 1:
        raise ValueError(f"Attach one input containing {marker}. For Stage 3 progress, attach only the latest snapshot.")
    if roots:
        return roots[0]
    root = Path(tempfile.mkdtemp(prefix="nerf-dense-"))
    with zipfile.ZipFile(archives[0]) as archive:
        if marker not in archive.namelist() or any(not (root / n).resolve().is_relative_to(root) for n in archive.namelist()):
            raise ValueError(f"Unexpected layout in {archives[0].name}")
        archive.extractall(root)
    return root

repo = mounted("RECOVERY_STAGE0.txt", ["stage0-input.zip"])
base = mounted("stage2-review.json", ["lego-dense-vanilla-s42-step2000.zip"])
later = mounted("stage3-review.json", ["lego-dense-vanilla-s42-stage3-*.zip",
                                       "lego-dense-vanilla-s42-step50000.zip"], optional=True)
work.mkdir(parents=True, exist_ok=True)
hardware = dict(torch=str(torch.__version__), cuda=torch.version.cuda,
                devices=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
(work / "stage3-hardware.json").write_text(json.dumps(hardware, indent=2))
print(hardware)
assert torch.cuda.is_available(), "Select T4 in Kaggle Notebook settings."

In [ ]:
sys.path.insert(0, str(repo))
from recovery.training import implementation_hash, load_state, settings

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

base_run = base / relative_run
meta = read_json(base_run / "run.json")
stage2 = read_json(base / "stage2-review.json")
assert implementation_hash() == code_hash
assert stage2["complete"] and stage2["run_id"] == meta["run_id"] == run_id
assert stage2["start_step"] == 500 and stage2["checkpoint_step"] == 2000
assert stage2["checkpoint_sha256"] == sha(base_run / "checkpoint.pt") == source_hash
assert meta["protocol"] == "dense-vanilla" and meta["scene"] == "lego" and meta["seed"] == 42
assert meta["train_views"] == 100 and meta["data_hash"] == data_hash

subprocess.run([sys.executable, "-m", "recovery.data", "--verify", str(base / "manifests/lego.json"),
                "--source", str(repo / "data/blender/lego"), "--output", str(manifest)], cwd=repo, check=True)
frozen = read_json(manifest)
assert frozen["data_hash"] == data_hash and meta["training_config"] == settings("main", frozen["camera"])
assert meta["training_config"]["steps"] == 50_000

def step_at(folder):
    state = load_state(folder, meta)
    if not state["optimizer"]["state"] or state.get("cuda_rng") is None or state.get("cpu_rng") is None:
        raise ValueError("Expected saved Adam and RNG states")
    return int(state["step"])

assert step_at(base_run) == 2000
source = base_run
if later is not None:
    saved = read_json(later / "stage3-review.json")
    source = later / relative_run
    assert saved["run_id"] == run_id and saved["source_checkpoint_sha256"] == source_hash
    assert saved["checkpoint_sha256"] == sha(source / "checkpoint.pt")
    assert 2000 <= saved["checkpoint_step"] == step_at(source) <= 50_000
assert read_json(source / "run.json") == meta
run.parent.mkdir(parents=True, exist_ok=True)
if not run.exists():
    with tempfile.TemporaryDirectory(dir=run.parent) as tmp:
        staged = Path(tmp) / "seed-42"
        shutil.copytree(source, staged, ignore=shutil.ignore_patterns(".train.lock", "*.tmp"))
        assert sha(staged / "checkpoint.pt") == sha(source / "checkpoint.pt")
        staged.rename(run)
assert read_json(run / "run.json") == meta
start = step_at(run)
assert step_at(source) <= start <= 50_000, "Use a fresh session for the selected input snapshot."
if start == step_at(source):
    assert sha(run / "checkpoint.pt") == sha(source / "checkpoint.pt")
os.chdir(outputs)
print(f"Ready at {start:,} of 50,000 updates. Run {run_id}.")

In [ ]:
STOP_AT = 50_000  # Absolute stop. Use e.g. 10_000 for a shorter session and rerun for the rest.
start = step_at(run)
assert start <= STOP_AT <= 50_000 and isinstance(STOP_AT, int)
log_path = work / "dense-finish.log"

def train_until(end):
    cmd = [sys.executable, "-u", "-m", "recovery.training", "--preset", "main", "--protocol", "dense-vanilla",
           "--seed", "42", "--manifest", str(manifest), "--run", str(run), "--device", "cuda:0",
           "--resume", "--save-every", "500", "--stop-at", str(end)]
    with log_path.open("a", encoding="utf-8") as log:
        with subprocess.Popen(cmd, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
            try:
                for line in proc.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                if proc.wait():
                    raise RuntimeError(f"Training stopped with status {proc.returncode}. See dense-finish.log.")
            except BaseException:
                if proc.poll() is None:
                    proc.terminate()
                    proc.wait()
                raise

def snapshot(step, reached):
    final = reached and step == 50_000
    tag = "step50000" if final else f"stage3-{step:05d}" + ("" if reached else "-interrupted")
    archive = outputs / f"lego-dense-vanilla-s42-{tag}.zip"
    digest = sha(run / "checkpoint.pt")
    record = dict(run_id=run_id, source_checkpoint_sha256=source_hash, checkpoint_sha256=digest,
                  checkpoint_step=step, complete=final, planned_stop_reached=reached)
    if archive.exists():
        with zipfile.ZipFile(archive) as previous:
            original = json.loads(previous.read("stage3-review.json"))
            if original != record or previous.testzip() is not None:
                raise ValueError(f"Preserve {archive.name} separately before writing a different snapshot.")
    else:
        pending = archive.with_suffix(".tmp")
        with zipfile.ZipFile(pending, "w", compression=zipfile.ZIP_STORED) as z:
            for p in [manifest, work / "stage3-hardware.json", log_path, *sorted(run.rglob("*"))]:
                if p.is_file() and not p.name.startswith(".") and not p.name.endswith(".tmp"):
                    z.write(p, p.relative_to(work))
            z.writestr("stage3-review.json", json.dumps(record, indent=2))
        os.replace(pending, archive)
    print(f"Update {step:,} saved. Download {archive.name} from Output.", flush=True)
    display(FileLink(archive.name))

ends = [end for end in range(10_000, STOP_AT, 10_000) if start < end] + [STOP_AT]
for end in ends:
    reached = False
    try:
        if step_at(run) < end:
            train_until(end)
        assert step_at(run) == read_json(run / "health.json")["step"] == end
        last = json.loads((run / "training.jsonl").read_text().splitlines()[-1])
        assert last["step"] == end and (run / f"train-{end:06d}.png").exists()
        reached = True
        print(f"At {end:,} of 50,000. Last segment: {last['steps_per_s']:.2f} updates/s.")
    finally:
        snapshot(step_at(run), reached)